# SynBO Multi-Objective Reaction Optimization Demo

This notebook demonstrates a complete **multi-objective reaction optimization**
workflow using SynBO on a cobalt-catalyzed asymmetric reaction.

- **Reaction space**: 5 reagent types x ~46k combinations
- **Objectives**: Maximize **yield** and **enantiomeric excess (ee)**
- **Method**: Bayesian Optimization with explore/exploit trade-off


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from synbo import ReactionOptimizer
from synbo.utils import load_desc_dict, get_prev_rxn
from virtual_experiment import virtual_experiment

sns.set_style('whitegrid')
%matplotlib inline

print('SynBO imported successfully')


## Step 1: Load the Reaction Space

The reaction space defines all possible reagent choices across 5 condition
types. Each CSV file in `rxn_space/` lists the available reagents with
SMILES strings, while the corresponding `descriptors/` directory provides
pre-computed RDKit molecular descriptors.


In [ ]:
EXAMPLE_DIR = Path('.').resolve()

REAGENT_TYPES = [
    'alkali',
    'cobalt_catalyst',
    'organo_catalyst',
    'oxidant',
    'solvent',
]

desc_dict, condition_dict = load_desc_dict(
    reagent_types=REAGENT_TYPES,
    desc_dir=str(EXAMPLE_DIR / 'descriptors'),
    name_suffix='_RDKit',
    index_col='name',
    return_condition_dict=True,
)

print('Reaction Space Summary')
print('-' * 50)
total = 1
for rtype, values in condition_dict.items():
    count = len(values)
    total *= count
    print(f'  {rtype:25s}: {count:3d} options')
print(f'  Total combinations: {total:,}')
print(f'  Descriptor dimensions: {sum(df.shape[1] for df in desc_dict.values())}')


## Step 2: Initial Sampling - Batch 0

The first batch uses **Latin Hypercube Sampling (LHS)** to generate a
space-filling initial design. Since no prior data exists, every
recommendation is tagged as `EXPLORE`.


In [ ]:
optimizer = ReactionOptimizer(
    opt_metrics=['yield', 'ee'],
    opt_type='init',
    random_seed=42,
    save_dir=str(EXAMPLE_DIR / 'results'),
)
optimizer.load_rxn_space(condition_dict)
optimizer.load_desc(desc_dict)
optimizer.initialize(batch_size=8, sampling_method='lhs')
optimizer.save_results(filetype='csv')

print('\nBatch 0 recommendations (8 initial conditions):')
for i, cond in enumerate(optimizer.selected_conditions):
    d = dict(zip(REAGENT_TYPES, cond))
    print(f'  {i + 1}. {d}')


## Step 3: Simulate Experiments for Batch 0

In a real workflow you would run wet-lab experiments and record yield/ee.
Here we call `virtual_experiment()` to **randomly populate the saved CSV
file on disk** with simulated outcomes, then read it back so the notebook
always uses the on-disk data.


In [ ]:
# Locate the CSV that optimizer.save_results() just wrote
batch0_files = sorted((EXAMPLE_DIR / 'results').glob('batch-0_*.csv'))
batch0_file = batch0_files[-1]

# Simulate experiments - writes to disk, returns the updated DataFrame
batch0_data = virtual_experiment(batch0_file, seed=42)

print(f'Updated {batch0_file.name} with simulated results')
print('\nFirst 8 experimental outcomes:')
display(batch0_data[['alkali', 'cobalt_catalyst', 'organo_catalyst', 'yield', 'ee']].round(1))


## Step 4: Bayesian Optimization - Batch 1

With experimental results from Batch 0, we train a **surrogate model**
(Gaussian Process) and use an acquisition function to recommend the next
batch. SynBO balances **exploitation** (high predicted performance) and
**exploration** (high uncertainty).


In [ ]:
optimizer = ReactionOptimizer(
    opt_metrics=['yield', 'ee'],
    opt_type='auto',
    random_seed=42,
    save_dir=str(EXAMPLE_DIR / 'results'),
)
optimizer.load_rxn_space(condition_dict)
optimizer.load_desc(desc_dict)
optimizer.load_prev_rxn(batch0_data)
optimizer.optimize(batch_size=5)
optimizer.save_results(filetype='csv')

print('Batch 1 recommendations:')
for i, (cond, rtype, mu, sigma) in enumerate(zip(
    optimizer.selected_conditions,
    optimizer.recommend_type,
    optimizer.pred_mean,
    optimizer.pred_std,
)):
    d = dict(zip(REAGENT_TYPES, cond))
    print(f'  {i + 1}. [{rtype}] {d}')
    print(f'       pred yield: {mu[0]:.1f} +/- {sigma[0]:.1f}  |  pred ee: {mu[1]:.1f} +/- {sigma[1]:.1f}')

n_explore = sum(1 for t in optimizer.recommend_type if t.lower() == 'explore')
n_exploit = sum(1 for t in optimizer.recommend_type if t.lower() == 'exploit')
print(f'\n  -> Explore: {n_explore}  |  Exploit: {n_exploit}')


## Step 5: Multi-Round Optimization Loop

We now run **6 additional rounds** of the optimization cycle:

1. **Recommend** - Bayesian optimization proposes the next batch
2. **Save** - write recommendations to CSV (with `[exp_data]` placeholders)
3. **Experiment** - `virtual_experiment()` fills in yield/ee on disk
4. **Reload** - read the updated CSV back from disk

This disk-based workflow mimics a real lab: results are persisted to
files and the notebook always reads the latest saved data.


In [ ]:
all_data = batch0_data.copy()
N_EXTRA_ROUNDS = 6

for n in range(1, N_EXTRA_ROUNDS + 1):
    optimizer = ReactionOptimizer(
        opt_metrics=['yield', 'ee'],
        opt_type='auto',
        random_seed=42 + n,
        save_dir=str(EXAMPLE_DIR / 'results'),
    )
    optimizer.load_rxn_space(condition_dict)
    optimizer.load_desc(desc_dict)
    optimizer.load_prev_rxn(all_data)
    optimizer.optimize(batch_size=5)

    # 1. Save recommendations (yield/ee = [exp_data] placeholder)
    optimizer.save_results(filetype='csv')

    # 2. Simulate experiments on disk
    batch_files = sorted(
        (EXAMPLE_DIR / 'results').glob(f'batch-{optimizer.batch_id}_*.csv')
    )
    batch_file = batch_files[-1]
    virtual_experiment(batch_file, seed=42 + n)

    # 3. Reload from disk (always reads the saved file)
    new_data = pd.read_csv(batch_file)

    all_data = pd.concat([all_data, new_data], ignore_index=True)
    print(
        f'Round {n}: +{len(new_data)} experiments  ->  '
        f'total = {len(all_data)}'
    )

print(f'\nFinished: {len(all_data)} experiments across {N_EXTRA_ROUNDS + 1} batches')


## Step 6: Visualize Optimization Progress

The **Pareto front** shows the trade-off between yield and ee. Points on
the front are *non-dominated* (you cannot improve one objective without
hurting the other). We also track best-per-batch values and a simple
hypervolume proxy.


In [ ]:
def get_pareto_mask(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    pts = np.column_stack([x, y])
    is_pareto = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        for j in range(len(pts)):
            if np.all(pts[j] >= pts[i]) and np.any(pts[j] > pts[i]):
                is_pareto[i] = False
                break
    return is_pareto


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

batch_ids = sorted(all_data['batch'].unique())
cmap = plt.cm.viridis

# Left: scatter by batch + Pareto front
for bid in batch_ids:
    color = cmap(bid / batch_ids[-1])
    sub = all_data[all_data['batch'] == bid]
    axes[0].scatter(
        sub['yield'], sub['ee'],
        color=color, label=f'Batch {int(bid)}', alpha=0.7, s=60,
    )

mask = get_pareto_mask(all_data['yield'].values, all_data['ee'].values)
pareto = all_data[mask].sort_values('yield')
axes[0].plot(pareto['yield'], pareto['ee'], 'r--', lw=2, label='Pareto front')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].set_xlabel('Yield (%)')
axes[0].set_ylabel('ee (%)')
axes[0].set_title('Optimization Progress by Batch')

# Middle: best-per-batch line chart
best_by_batch = all_data.groupby('batch').agg({'yield': 'max', 'ee': 'max'})
axes[1].plot(best_by_batch.index, best_by_batch['yield'], 'o-', lw=2, label='Best yield')
axes[1].plot(best_by_batch.index, best_by_batch['ee'], 's-', lw=2, label='Best ee')
axes[1].set_xlabel('Batch')
axes[1].set_ylabel('Value (%)')
axes[1].set_title('Best Values Over Time')
axes[1].legend()

# Right: hypervolume bar chart
hv_vals = []
for bid in batch_ids:
    sub = all_data[all_data['batch'] <= bid]
    m = get_pareto_mask(sub['yield'].values, sub['ee'].values)
    pf = sub[m]
    hv_vals.append(np.prod(np.maximum(pf[['yield', 'ee']].values / 100, 0).max(axis=0)))

colors = [cmap(b / batch_ids[-1]) for b in batch_ids]
axes[2].bar(batch_ids, hv_vals, color=colors)
axes[2].set_xlabel('Batch')
axes[2].set_ylabel('Hypervolume')
axes[2].set_title('Hypervolume Progress')

plt.tight_layout()
plt.show()

print(f'Pareto-optimal points: {mask.sum()}')
print(f'Best yield: {all_data["yield"].max():.1f}%')
print(f'Best ee:    {all_data["ee"].max():.1f}%')


## Step 7: Final Pareto Front

A cleaner view of the final Pareto front together with the ideal (but
unattainable) target of 100% yield and 100% ee.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    all_data['yield'], all_data['ee'],
    c=all_data['batch'], cmap='viridis', s=80, alpha=0.7,
    edgecolors='k', linewidth=0.5,
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Batch')

mask = get_pareto_mask(all_data['yield'].values, all_data['ee'].values)
pareto = all_data[mask].sort_values('yield')
ax.plot(pareto['yield'], pareto['ee'], 'r-', lw=2.5, label='Pareto Front', zorder=5)
ax.scatter(
    pareto['yield'], pareto['ee'],
    color='red', s=120, zorder=6, edgecolors='darkred',
)

ax.scatter(
    [100], [100], marker='*', color='gold', s=400,
    edgecolors='black', zorder=7, label='Ideal (100%, 100%)',
)

ax.set_xlabel('Yield (%)', fontsize=12)
ax.set_ylabel('ee (%)', fontsize=12)
ax.set_title('Pareto Front: Yield vs Enantiomeric Excess', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()


## Summary

This notebook demonstrated an end-to-end SynBO workflow:

| Step | Description |
|:-----|:------------|
| **1. Reaction space** | 5 reagent types loaded from CSV descriptors |
| **2. Initial sampling** | LHS generates a space-filling Batch 0 (8 conditions) |
| **3. Simulate experiments** | `virtual_experiment()` writes random yield/ee to disk |
| **4. Bayesian optimization** | GP surrogate + acquisition function for Batch 1 |
| **5. Multi-round loop** | 6 more batches with disk-based experiment simulation |
| **6. Visualization** | Pareto front, best-value tracking, hypervolume |

### Key Design Points

- **`virtual_experiment` lives in `virtual_experiment.py`** - no simulation
  logic is hard-coded in the notebook.
- **All data is read from disk** after each round, mimicking a real lab
  workflow where CSV files are the source of truth.
- **Explore/exploit balance** is handled automatically by SynBO's acquisition
  function.
